In [7]:
import numpy as np

In [ ]:
# Verificaiton
# -------------------
# 1) Column/Material Properties and assumptions(Inputs)
# -------------------
# E   = 29000.0       # ksi (typical steel)
# For a W14x120, strong-axis I is ~650 to 660 in^4 (depends on the exact table).
# I1L  = 657.0         # in^4 (example)
# I1R  = 657.0         # in^4 (example)
# I2L  = 657.0         # in^4 (assuming same cross-section)
# I2R  = 657.0         # in^4 (assuming same cross-section)
# I3L  = 657.0         # in^4 (assuming same cross-section)
# I3R  = 657.0         # in^4 (assuming same cross-section)

EI1L = 152000 # scale??????????????????????????????????????????????????????????????
EI1R = 152000
EI2L = 152000
EI2R = 152000
EI3L = 152000
EI3R = 152000


# Story heights (inches):
h1 = 4 #m
h2 = 4
h3 = 4


# Dead Mass
m1_dead = 45000   # kg
m2_dead = 45000
m3_dead = 45000/2

# Coefficient for live to dead load
coeff = 1

# live to dead
m1_live = 0 #m1_dead / coeff
m2_live = 0 #m2_dead / coeff
m3_live = 0 #m3_dead / coeff


# We will vary the live-load portion alpha in [0.25, 0.75]
num_point = 1
max_live_portion = 1
min_live_portion = 1

# Compute each story stiffness (2 columns in parallel), I put stiffness here because we can change the equation if frame changes
k1L = 12.0 * EI1L / (h1**3)
k1R = 12.0 * EI1R / (h1**3)
k2L = 12.0 * EI2L / (h2**3)
k2R = 12.0 * EI2R / (h2**3)
k3L = 12.0 * EI3L / (h3**3)
k3R = 12.0 * EI3R / (h3**3)

In [ ]:
# # Example
# # -------------------
# # 1) Column/Material Properties and assumptions(Inputs)
# # -------------------
# E   = 29000.0       # ksi (typical steel)
# # For a W14x120, strong-axis I is ~650 to 660 in^4 (depends on the exact table).
# I1L  = 657.0         # in^4 (example)
# I1R  = 657.0         # in^4 (example)
# I2L  = 657.0         # in^4 (assuming same cross-section)
# I2R  = 657.0         # in^4 (assuming same cross-section)
# I3L  = 657.0         # in^4 (assuming same cross-section)
# I3R  = 657.0         # in^4 (assuming same cross-section)

# # Story heights (inches):
# h1 = 180.0  # e.g., 15 ft = 180 in
# h2 = 180.0
# h3 = 180.0

# # Dead Mass
# m1_dead = 0.24   # kip*s^2/in
# m2_dead = 0.24
# m3_dead = 0.12

# # Coefficient for live to dead load
# coeff = 3

# # live to dead
# m1_live = m1_dead / coeff
# m2_live = m2_dead / coeff
# m3_live = m3_dead / coeff

# # We will vary the live-load portion alpha in [0.25, 0.75]
# num_point = 6
# max_live_portion = 0.75
# min_live_portion = 0.25

# # Compute each story stiffness (2 columns in parallel), I put stiffness here because we can change the equation if frame changes
# k1L = 12.0 * E * I1L / (h1**3)
# k1R = 12.0 * E * I1R / (h1**3)
# k2L = 12.0 * E * I2L / (h2**3)
# k2R = 12.0 * E * I2R / (h2**3)
# k3L = 12.0 * E * I3L / (h3**3)
# k3R = 12.0 * E * I3R / (h3**3)

In [ ]:
def build_mass_matrix(m1, m2, m3):
    """
    Returns a 3x3 diagonal mass matrix [M].
    """
    return np.diag([m1, m2, m3])

def build_stiffness_matrix(k1L, k1R, k2L, k2R, k3L, k3R):
    """
    Returns a 3x3 stiffness matrix [K] for a 3-DOF shear building.
    """
    # First floor
    k1 = k1L+k1R
    # Second floor
    k2 = k2L+k2R
    # Third floor
    k3 = k3L+k3R
    
    return np.array([
        [k1 + k2,   -k2,       0   ],
        [    -k2,   k2 + k3,   -k3 ],
        [     0,       -k3,    k3  ]
    ])

In [ ]:
# -------------------
# 2) Masses and live load variation
# -------------------
# We will vary the live-load portion alpha in [0.25, 0.75].
alpha_values = np.linspace(min_live_portion, max_live_portion, num_point)  # For example 6 points: 0.25, 0.35, ..., 0.75

# Prepare arrays to store results
freqs_array = []  # will store [alpha, f1, f2, f3] in each row

for alpha in alpha_values:
    # Compute total masses for this alpha:
    m1 = m1_dead + alpha*m1_live
    m2 = m2_dead + alpha*m2_live
    m3 = m3_dead + alpha*m3_live

    # Build M, K
    M = build_mass_matrix(m1, m2, m3)
    K = build_stiffness_matrix(k1L, k1R, k2L, k2R, k3L, k3R)

    print(M, K)

    # Solve the generalized eigenvalue problem: K phi = w^2 M phi
    # --> inv(M)*K * phi = w^2 phi
    # eigvals, eigvecs = np.linalg.eig(np.linalg.inv(M).dot(K))


    # Directly solve the generalized eigenvalue problem:
    # eig works for general matrices and therefore uses a slower algorithm in compared with eigh
    eigvals, eigvecs = np.linalg.eig(K, M)


    # # The eigenvalues can come out in any order, and might be complex due to
    # # numerical roundoff. We'll sort by magnitude and take real part:
    # idx = np.argsort(eigvals)
    # eigvals = eigvals[idx].real
    # eigvecs = eigvecs[:, idx].real

    # Compute circular frequencies and convert to natural freq in Hz
    w = np.sqrt(eigvals)           # rad/s
    f = w / (2.0*np.pi)            # Hz

    # Store the results:
    freqs_array.append([alpha, f[0], f[1], f[2]])

    # (Optional) Print out mode shapes (normalized so that the top entry = 1, etc.)
    # Let’s say we want to keep them consistent:
    # for each mode i, we can normalize so that the largest displacement component is 1.
    # NOTE: in practice, you can choose any scaling you like.
    mode_shapes = []
    for i in range(3):
        mode_i = eigvecs[:, i]
        mode_i /= np.max(np.abs(mode_i))  # largest component = 1
        mode_shapes.append(mode_i)

    print(f"alpha = {alpha:.2f}")
    print("Frequencies (Hz): ", f)
    print("Mode shapes (floor1, floor2, floor3) normalized:")
    for i, shape in enumerate(mode_shapes, start=1):
        print(f"  Mode {i}: {shape}")
    print("-"*50)

# Now `freqs_array` holds the frequencies for each alpha in the range.
# You can convert it to a NumPy array and do further analysis or plotting.

freqs_array = np.array(freqs_array)
print("Summary of frequencies vs. alpha:")
print(" alpha |   f1(Hz)    f2(Hz)    f3(Hz)")
for row in freqs_array:
    print(f"{row[0]:6.2f} | {row[1]:9.4f} {row[2]:9.4f} {row[3]:9.4f}")


alpha = 0.25
Frequencies (Hz):  [1.43066592 3.90865197 5.33931789]
Mode shapes (floor1, floor2, floor3) normalized:
  Mode 1: [0.5       0.8660254 1.       ]
  Mode 2: [-1.00000000e+00  4.18064759e-16  1.00000000e+00]
  Mode 3: [-0.5        0.8660254 -1.       ]
--------------------------------------------------
alpha = 0.35
Frequencies (Hz):  [1.40915092 3.8498719  5.25902282]
Mode shapes (floor1, floor2, floor3) normalized:
  Mode 1: [0.5       0.8660254 1.       ]
  Mode 2: [-1.00000000e+00 -6.84105969e-16  1.00000000e+00]
  Mode 3: [ 0.5       -0.8660254  1.       ]
--------------------------------------------------
alpha = 0.45
Frequencies (Hz):  [1.38857827 3.79366638 5.18224464]
Mode shapes (floor1, floor2, floor3) normalized:
  Mode 1: [0.5       0.8660254 1.       ]
  Mode 2: [-1.00000000e+00 -1.14017661e-16  1.00000000e+00]
  Mode 3: [-0.5        0.8660254 -1.       ]
--------------------------------------------------
alpha = 0.55
Frequencies (Hz):  [1.36888112 3.73985278 5.1